# TFG - Evaluación de modelos sin la variable "Occupation"


In [1]:
# 1. CARGA DE DATOS
import pandas as pd

# Cargar el dataset original
ruta = r"C:\Users\Marina\OneDrive - Universidad de Burgos\Escritorio\TFG-Marina-Martin\datos\Sleep_health_and_lifestyle_dataset.csv"
df = pd.read_csv(ruta)

print("Tamaño del dataset original:", df.shape)

Tamaño del dataset original: (374, 13)


In [2]:
# 2. LIMPIEZA DE "Occupation"

# Eliminar la columna original 'Occupation' si está presente
if 'Occupation' in df.columns:
    df = df.drop(columns=['Occupation'])

# Eliminar columnas codificadas tipo 'Occupation_*' (por si se ha hecho One-Hot antes)
df = df.loc[:, ~df.columns.str.startswith('Occupation_')]

# Comprobación final
print("Columnas con 'Occupation':", [col for col in df.columns if "Occupation" in col])
print("Tamaño del dataset tras limpieza:", df.shape)


Columnas con 'Occupation': []
Tamaño del dataset tras limpieza: (374, 12)


In [3]:
# 3. TRANSFORMACIÓN DE VARIABLES

from sklearn.preprocessing import StandardScaler, LabelEncoder

# Separar características y etiquetas
X = df.drop("Sleep Disorder", axis=1)
y = df["Sleep Disorder"]

# Variables categóricas a codificar (ya hemos eliminado 'Occupation')
cat_cols = ['Gender', 'BMI Category', 'Blood Pressure']

# Codificación One-Hot de variables categóricas
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Escalado de variables numéricas
num_cols = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level',
            'Stress Level', 'Heart Rate', 'Daily Steps']

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Codificación del target
le = LabelEncoder()
y = le.fit_transform(y)

# Verificar shapes
print("Shape de X:", X.shape)
print("Clases del target:", le.classes_)


C:\Users\Marina\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Shape de X: (374, 36)
Clases del target: ['Insomnia' 'None' 'Sleep Apnea']


In [4]:
# 4. SPLIT SIN SMOTE

from sklearn.model_selection import train_test_split

# División en train y test SIN aplicar SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Comprobación de distribución de clases
print("Distribución en y_train:", pd.Series(y_train).value_counts())
print("Distribución en y_test:", pd.Series(y_test).value_counts())


Distribución en y_train: 1    175
2     62
0     62
dtype: int64
Distribución en y_test: 1    44
2    16
0    15
dtype: int64


In [5]:
# 5. RANDOM FOREST MODEL

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
import numpy as np

# Definir el modelo base
rf = RandomForestClassifier(class_weight='balanced', random_state=42)

# Espacio de búsqueda de hiperparámetros
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Randomized Search
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Entrenar
rf_random.fit(X_train, y_train)

# Evaluar
y_pred_rf = rf_random.best_estimator_.predict(X_test)
print("Mejores hiperparámetros RF:", rf_random.best_params_)
print("Reporte clasificación RF:\n", classification_report(y_test, y_pred_rf, target_names=le.classes_))


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores hiperparámetros RF: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20, 'bootstrap': True}
Reporte clasificación RF:
               precision    recall  f1-score   support

    Insomnia       0.93      0.87      0.90        15
        None       1.00      1.00      1.00        44
 Sleep Apnea       0.88      0.94      0.91        16

    accuracy                           0.96        75
   macro avg       0.94      0.93      0.94        75
weighted avg       0.96      0.96      0.96        75



In [7]:
import joblib

from sklearn.calibration import CalibratedClassifierCV

# Calibrar el mejor modelo encontrado
calibrated_rf = CalibratedClassifierCV(estimator=rf_random.best_estimator_, cv="prefit")
calibrated_rf.fit(X_train, y_train)

# Guardar el modelo calibrado
joblib.dump(calibrated_rf, "rf_sin_ocupacion.joblib")

['rf_sin_ocupacion.joblib']